In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# USER SETTINGS
# ============================================================

sense_resistor_ohm = 99_200

# Analyze BOTH data folders
root_folders = [
    Path(r"C:\Users\mtpv1\Downloads\EIS-Data_8_13_26"),
    Path(r"C:\Users\mtpv1\Downloads\EIS-Data_9_9_26")
]

# Save the combined CSV here
output_folder = Path(r"C:\Users\mtpv1\Downloads\EIS-Data_Combined")

# Create output folder if it does not exist
output_folder.mkdir(parents=True, exist_ok=True)

# ============================================================
# LOAD TEKTRONIX CSV
# ============================================================

def load_tektronix_csv(filename):

    df = pd.read_csv(
        filename,
        header=None,
        usecols=[3, 4],
        names=["time_s", "voltage_v"]
    )

    df["time_s"] = pd.to_numeric(
        df["time_s"],
        errors="coerce"
    )

    df["voltage_v"] = pd.to_numeric(
        df["voltage_v"],
        errors="coerce"
    )

    df = df.dropna().reset_index(drop=True)

    return df

# ============================================================
# SINE FIT
# ============================================================

def fit_sine(df, frequency_hz):

    t = df["time_s"].to_numpy()
    v = df["voltage_v"].to_numpy()

    omega = 2 * np.pi * frequency_hz

    X = np.column_stack([
        np.sin(omega * t),
        np.cos(omega * t),
        np.ones_like(t)
    ])

    coefficients, _, _, _ = np.linalg.lstsq(
        X,
        v,
        rcond=None
    )

    a, b, offset = coefficients

    amplitude_peak = np.sqrt(a**2 + b**2)
    amplitude_rms = amplitude_peak / np.sqrt(2)

    phase_deg = np.degrees(
        np.arctan2(b, a)
    )

    fitted_v = X @ coefficients
    residual = v - fitted_v

    residual_rms = np.sqrt(
        np.mean(residual**2)
    )

    snr_linear = amplitude_rms / residual_rms
    snr_db = 20 * np.log10(snr_linear)

    ss_res = np.sum(residual**2)
    ss_tot = np.sum(
        (v - np.mean(v))**2
    )

    r_squared = 1 - ss_res / ss_tot

    return {
        "rms_v": amplitude_rms,
        "phase_deg": phase_deg,
        "offset_v": offset,
        "residual_rms_v": residual_rms,
        "snr_db": snr_db,
        "r_squared": r_squared
    }

# ============================================================
# ANALYZE ONE FREQUENCY
# ============================================================

def analyze_pair(
    voltage_file,
    sense_file,
    frequency_hz,
    sense_resistor_ohm
):

    voltage_df = load_tektronix_csv(
        voltage_file
    )

    sense_df = load_tektronix_csv(
        sense_file
    )

    voltage_fit = fit_sine(
        voltage_df,
        frequency_hz
    )

    sense_fit = fit_sine(
        sense_df,
        frequency_hz
    )

    # Current through sense resistor
    current_rms = (
        sense_fit["rms_v"]
        / sense_resistor_ohm
    )

    voltage_phasor = (
        voltage_fit["rms_v"]
        * np.exp(
            1j * np.radians(
                voltage_fit["phase_deg"]
            )
        )
    )

    sense_phasor = (
        sense_fit["rms_v"]
        * np.exp(
            1j * np.radians(
                sense_fit["phase_deg"]
            )
        )
    )

    current_phasor = (
        sense_phasor
        / sense_resistor_ohm
    )

    current_rms = abs(current_phasor)

    cell_voltage_phasor = (
        voltage_phasor
        - sense_phasor
    )

    cell_impedance = (
        cell_voltage_phasor
        / current_phasor
    )

    impedance_ohm = abs(cell_impedance)

    phase_deg = np.degrees(
        np.angle(cell_impedance)
    )

    return {
        "Frequency_Hz": frequency_hz,

        "Applied_Vrms_mV":
            voltage_fit["rms_v"] * 1000,

        "Cell_Vrms_mV":
            abs(cell_voltage_phasor) * 1000,

        "Sense_Vrms_mV":
            sense_fit["rms_v"] * 1000,

        "Current_nA":
            current_rms * 1e9,

        "Impedance_Ohm":
            impedance_ohm,

        "Phase_deg":
            phase_deg,

        "Sense_SNR_dB":
            sense_fit["snr_db"],

        "Sense_R2":
            sense_fit["r_squared"],

        "Sense_Residual_mV":
            sense_fit["residual_rms_v"] * 1000
    }

# ============================================================
# ANALYZE ALL ROOT FOLDERS
# ============================================================

all_results = []

for root_folder in root_folders:

    print("\n" + "=" * 60)
    print(f"DATA FOLDER: {root_folder}")
    print("=" * 60)

    if not root_folder.exists():
        print(f"Folder does not exist: {root_folder}")
        continue

    for electrode_folder in root_folder.iterdir():

        if not electrode_folder.is_dir():
            continue

        print(f"\nAnalyzing {electrode_folder.name}")

        # ----------------------------------------------------
        # Split folder metadata
        # ----------------------------------------------------

        parts = electrode_folder.name.split("_")

        if len(parts) < 4:
            print(
                f"Unexpected electrode folder name: "
                f"{electrode_folder.name}"
            )
            continue

        wafer = parts[0]
        device = parts[1]
        material = parts[2]
        size = parts[3]

        folder = electrode_folder

        # ====================================================
        # FIND ALL FILE PAIRS
        # ====================================================

        for voltage_file in folder.glob("*_1.csv"):

            # Example:
            # 4_1_Al_100_1000_1.csv
            #
            # Second-to-last field = frequency
            # Last field = channel

            file_parts = voltage_file.stem.split("_")

            if len(file_parts) < 2:
                print(
                    f"Unexpected filename: "
                    f"{voltage_file.name}"
                )
                continue

            frequency_string = file_parts[-2]

            try:
                frequency_hz = float(
                    frequency_string
                )

            except ValueError:
                print(
                    f"Could not determine frequency from "
                    f"{voltage_file.name}"
                )
                continue

            base_name = voltage_file.stem

            if base_name.endswith("_1"):
                base_name = base_name[:-2]

            sense_filename = (
                base_name + "_2.csv"
            )

            sense_file = (
                folder / sense_filename
            )

            if not sense_file.exists():

                print(
                    f"Missing pair for "
                    f"{voltage_file.name}"
                )
                continue

            print(
                f"  Analyzing {frequency_hz:g} Hz..."
            )

            result = analyze_pair(
                voltage_file,
                sense_file,
                frequency_hz,
                sense_resistor_ohm
            )

            result["Wafer"] = wafer
            result["Device"] = device
            result["Material"] = material
            result["Size"] = size

            # Useful so you know which experiment
            # each row came from
            result["Data_Folder"] = root_folder.name

            all_results.append(result)

# ============================================================
# CREATE ONE MASTER DATAFRAME
# ============================================================

results_df = pd.DataFrame(all_results)

if len(results_df) == 0:
    print("\nNo valid EIS data found.")

else:

    results_df = results_df.sort_values(
        [
            "Wafer",
            "Device",
            "Material",
            "Size",
            "Frequency_Hz",
            "Data_Folder"
        ]
    )

    # ========================================================
    # SAVE ONE COMBINED CSV
    # ========================================================

    output_file = (
        output_folder
        / "All_EIS_results.csv"
    )

    results_df.to_csv(
        output_file,
        index=False
    )

    print(
        f"\nAll results saved to:\n"
        f"{output_file}"
    )


DATA FOLDER: C:\Users\mtpv1\Downloads\EIS-Data_8_13_26

Analyzing 4_1_Al_100
  Analyzing 10000 Hz...
  Analyzing 1000 Hz...
  Analyzing 100 Hz...
  Analyzing 10 Hz...
  Analyzing 1 Hz...
  Analyzing 5000 Hz...
  Analyzing 500 Hz...
  Analyzing 50 Hz...
  Analyzing 5 Hz...

Analyzing 4_1_Al_150
  Analyzing 10000 Hz...
  Analyzing 1000 Hz...
  Analyzing 100 Hz...
  Analyzing 10 Hz...
  Analyzing 1 Hz...

Analyzing 4_1_Al_200
  Analyzing 10000 Hz...
  Analyzing 1000 Hz...
  Analyzing 100 Hz...
  Analyzing 10 Hz...
  Analyzing 1 Hz...

Analyzing 4_1_Al_250
  Analyzing 10000 Hz...
  Analyzing 1000 Hz...
  Analyzing 100 Hz...
  Analyzing 10 Hz...
  Analyzing 1 Hz...

Analyzing 4_1_Al_300
  Analyzing 10000 Hz...
  Analyzing 1000 Hz...
  Analyzing 100 Hz...
  Analyzing 10 Hz...
  Analyzing 1 Hz...

Analyzing 4_1_Al_50
  Analyzing 10000 Hz...
  Analyzing 1000 Hz...
  Analyzing 100 Hz...
  Analyzing 10 Hz...
  Analyzing 1 Hz...
  Analyzing 5000 Hz...
  Analyzing 500 Hz...
  Analyzing 50 Hz...
 